<a href="https://colab.research.google.com/github/semal-1820/AI-VFX-Agent-Pipeline/blob/main/Visual_FX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers diffusers accelerate transformers[torch] diffusers[torch] opencv-python pillow

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import SamModel, SamProcessor
from diffusers import StableDiffusionInpaintPipeline

class AIVFXPipeline:
    def __init__(self):
        # Determine if a GPU is available (Crucial for Colab T4)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")

        # 1. Load Object Detection Model (OWLv2)
        print("Loading Object Detection Model (OWLv2)...")
        self.det_processor = AutoProcessor.from_pretrained("google/owlv2-base-patch16")
        self.det_model = AutoModelForZeroShotObjectDetection.from_pretrained("google/owlv2-base-patch16").to(self.device)

        # 2. Load Segmentation Model (SAM)
        print("Loading Segmentation Model (SAM)...")
        self.sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
        self.sam_model = SamModel.from_pretrained("facebook/sam-vit-base").to(self.device)

        # 3. Load Generative Inpainting Model (Stable Diffusion Inpaint)
        print("Loading Inpainting Model (Stable Diffusion)...")
        self.inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "runwayml/stable-diffusion-inpainting",
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        ).to(self.device)

    def detect_object(self, image: Image.Image, text_queries: list):
        """Locates the bounding box of the target text prompt in the image."""
        inputs = self.det_processor(text=text_queries, images=image, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.det_model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]]).to(self.device)

        # BUG FIX: Corrected the processor method name here!
        results = self.det_processor.post_process_object_detection(
            outputs=outputs, threshold=0.1, target_sizes=target_sizes
        )

        if len(results[0]["boxes"]) > 0:
            best_box = results[0]["boxes"][0].cpu().numpy().astype(int).tolist()
            return best_box
        else:
            raise ValueError(f"Could not locate '{text_queries[0]}' in the image. Try a lower threshold or a different prompt.")

    def generate_mask(self, image: Image.Image, bbox: list):
        """Uses SAM to turn a bounding box into a highly detailed binary mask."""
        input_boxes = [[bbox]]
        inputs = self.sam_processor(image, input_boxes=input_boxes, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.sam_model(**inputs)

        masks = self.sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), inputs["reshaped_input_sizes"].cpu()
        )

        mask_np = masks[0][0][0].numpy().astype(np.uint8) * 255
        return Image.fromarray(mask_np)

    def inpaint_scene(self, original_image: Image.Image, mask_image: Image.Image, prompt: str):
        """Inpaints the masked region with the new generated background/object."""
        w, h = original_image.size
        w, h = (w // 8) * 8, (h // 8) * 8  # Math constraint: SD dimensions must be multiples of 8

        img_resized = original_image.resize((w, h))
        mask_resized = mask_image.resize((w, h))

        output = self.inpaint_pipe(
            prompt=prompt,
            image=img_resized,
            mask_image=mask_resized,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]

        return output.resize((w, h))

    def run_vfx_pipeline(self, image_path: str, target_to_replace: str, new_prompt: str):
        """Orchestrates the entire end-to-end pipeline execution."""
        print("\n--- Starting Pipeline Execution ---")
        original_image = Image.open(image_path).convert("RGB")

        print(f"Step 1: Locating '{target_to_replace}' in the frame...")
        bbox = self.detect_object(original_image, text_queries=[target_to_replace])
        print(f"Found object at bounding box coordinates: {bbox}")

        print("Step 2: Generating pixel-perfect mask using SAM...")
        mask_image = self.generate_mask(original_image, bbox)

        print(f"Step 3: Replacing masked area with '{new_prompt}'...")
        final_output = self.inpaint_scene(original_image, mask_image, prompt=new_prompt)

        print("--- Process Complete! ---")
        return mask_image, final_output

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
# Initialize the pipeline class
vfx_agent = AIVFXPipeline()

Using device: cpu
Loading Object Detection Model (OWLv2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

The image processor of type `Owlv2ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

# Wrapper function to connect Gradio to your pipeline
def process_image_ui(input_image_path, target_object, generative_prompt):
    print(f"UI Request received: Replace '{target_object}' with '{generative_prompt}'")

    # Run the pipeline (vfx_agent is already loaded in Cell 3)
    mask, result = vfx_agent.run_vfx_pipeline(
        image_path=input_image_path,
        target_to_replace=target_object,
        new_prompt=generative_prompt
    )

    return mask, result

# Build the Web Interface
with gr.Blocks(theme=gr.themes.Soft()) as vfx_interface:
    gr.Markdown("# 🎬 AI VFX Agent Pipeline")
    gr.Markdown("Upload an image, specify the object you want to mask, and describe the new visual effect.")

    with gr.Row():
        # Left Column: User Inputs
        with gr.Column():
            img_input = gr.Image(type="filepath", label="1. Upload Original Image")
            target_input = gr.Textbox(
                label="2. Object to Replace",
                placeholder="e.g., 'sky', 'person', 'car'"
            )
            prompt_input = gr.Textbox(
                label="3. New VFX Prompt",
                placeholder="e.g., 'a vibrant cosmic sci-fi nebula'"
            )
            submit_btn = gr.Button("Generate VFX", variant="primary")

        # Right Column: AI Outputs
        with gr.Column():
            mask_output = gr.Image(label="Intermediate Step: SAM Mask")
            final_output = gr.Image(label="Final Output: Inpainted VFX")

    # Connect the button to the function
    submit_btn.click(
        fn=process_image_ui,
        inputs=[img_input, target_input, prompt_input],
        outputs=[mask_output, final_output]
    )

# Launch the app!
# share=True creates a temporary public link you can share in your documentation.
vfx_interface.launch(debug=True, share=True)

/tmp/ipykernel_7126/3875374912.py:17: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as vfx_interface:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6681cd13d16a3523f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
